In [82]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

model = 'claude-haiku-4-5-20251001'
max_tokens = 1000

client = Anthropic()

In [83]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [84]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete. Include a solution critera that provides a detailed set of things that are crucial to the task

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criteria": "Components that are crucial to successful task completion"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")  # Start the array
    
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [85]:
dataset = generate_dataset()

In [86]:
dataset

[{'task': "Create a JSON IAM policy that allows a user to read objects from a specific S3 bucket named 'my-data-bucket' and list bucket contents",
  'format': 'json',
  'solution_criteria': 'Must include: correct Version and Statement array structure, Principal or applicable IAM policy format, Action array with s3:GetObject and s3:ListBucket, Resource array specifying the bucket ARN and object ARN (bucket/* pattern), Effect set to Allow, and proper JSON syntax validation'},
 {'task': "Write a Python function that takes an AWS EC2 instance ID and returns the instance's current state (running, stopped, terminated, etc.) using boto3",
  'format': 'python',
  'solution_criteria': 'Must include: proper boto3 EC2 client initialization, describe_instances call with InstanceIds parameter, correct extraction of instance state from the response structure, error handling for invalid instance IDs, and return the state string value'},
 {'task': 'Create a regular expression that validates AWS S3 buc

In [87]:
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [88]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""Please solve the following task:

    {test_case["task"]}

* Respond with only Python, JSON or plan Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```code")
    output = chat(messages, stop_sequence = "```")
    return output

In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria:
<criteria>
Here are some critieria you should use to evaluate your solution
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "alignment": string,
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [90]:
def run_test_case(test_case):
    """Calls run_prompt, grades the result"""
    output = run_prompt(test_case)

    #Model Grader
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    #Code Grader
    syntax_score = grade_syntax(output,test_case)

    score = (model_score + syntax_score)/2

    return {
        "output":output,
        "test_case":test_case,
        "score":score,
        "reasoning":reasoning
    }

In [91]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [92]:
from statistics import mean

def  run_eval(dataset):
    """Loads the dataset and calls the run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    avg_score = mean([result["score"] for result in results])
    print(f"Average Score: {avg_score}")

    return results

In [93]:
with open("dataset.json",'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

TypeError: chat() got an unexpected keyword argument 'stop_sequence'

In [69]:
print(json.dumps(results,indent=2))

[
  {
    "output": "# Extract AWS Region from S3 Bucket URL\n\nHere's a solution using regex:\n\n```python\nimport re\n\ndef extract_aws_region(s3_url):\n    \"\"\"\n    Extract the AWS region from an S3 bucket URL.\n    \n    Examples:\n    - s3://my-bucket.s3.us-west-2.amazonaws.com/path/to/file -> us-west-2\n    - https://my-bucket.s3.eu-west-1.amazonaws.com/file -> eu-west-1\n    - s3://my-bucket/path/to/file -> None (uses default region)\n    \"\"\"\n    match = re.search(r'\\.s3[.-]([a-z0-9-]+)\\.amazonaws\\.com', s3_url)\n    return match.group(1) if match else None\n\n\n# Test cases\ntest_urls = [\n    's3://my-bucket.s3.us-west-2.amazonaws.com/path/to/file',\n    'https://my-bucket.s3.eu-west-1.amazonaws.com/file',\n    'https://my-bucket.s3.ap-southeast-1.amazonaws.com/key',\n    's3://my-bucket/path/to/file',  # No region specified\n    'https://my-bucket.s3.amazonaws.com/file',  # Default region\n]\n\nfor url in test_urls:\n    region = extract_aws_region(url)\n    print(f